# fptsim demo: load a finished run

First produce a run from this folder:

```bash
python -m fptsim run --config configs/example_small.yaml
```

Everything below imports the `fptsim` package; nothing is reimplemented here.

In [ ]:
import matplotlib.pyplot as plt

import fptsim
from fptsim.plots import plot_convergence, plot_distributions, plot_graph

run = fptsim.load_run("results/example_small")
run

In [ ]:
run.graphs[["graph_id", "family", "n", "num_edges", "k", "B", "dt", "q_mean", "spectral_gap"]].head(10)

In [ ]:
cols = ["graph_id", "replicate", "family", "n", "k", "frne_percent", "expected_rms_frne_percent",
        "ci_coverage_np5", "hit_mass_exact", "killed_mass_exact", "censored_mass_exact"]
ok = run.metrics[run.metrics.status == "ok"]
ok[cols].head(10)

In [ ]:
# replicate variability: mean / std of FRNE next to the expected RMS FRNE
(ok.groupby("graph_id")
   .agg(frne_mean=("frne_percent", "mean"), frne_std=("frne_percent", "std"),
        expected=("expected_rms_frne_percent", "first"), family=("family", "first"))
   .head(10))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, gid in zip(axes, run.graphs.graph_id[:3]):
    g = run.graph(int(gid))
    plot_graph(g, values=g.mu, label=r"$\mu$", ax=ax, title=f"graph {gid}: {g.family}, n={g.n}")
fig.tight_layout()

In [ ]:
gid = int(run.graphs.graph_id.iloc[0])
t = run.tensors(gid)
r_exact = t["sweeps"][0]["r_exact"]
r_emp = run.r_empirical(gid, s=0, r=0)
plot_distributions(r_exact, r_emp, t["B"], title=f"graph {gid}: $r'$ (bars) vs $r$ (line)");

In [ ]:
ax = plot_convergence(run.metrics)
if ax is None:
    print("no convergence checkpoints in this run (toggles.convergence_checkpoints: false)")